# LED on / off — Tang Nano 4K

Controls the Tang Nano 4K's LED (pin 10) from this notebook with nanopynq. Programming and
register access both go over the one USB cable (JTAG through the onboard debugger), so no
USB-UART adapter is needed.

Before you start:
- Windows: no driver change is needed with Windows' FTDI driver (nanopynq uses its D2XX; not yet checked on the 4K). Zadig (only "Interface 0" to WinUSB) also works.
- `python -m nanopynq doctor --board tangnano4k` shows no `NG`.

If a cell raises `BoardNotResponding`, unplug and re-plug the USB cable and run the `Overlay(...)`
cell again.

## 1. Program the board and open the link

`Overlay` writes the bundled base design to the FPGA (SRAM), then opens the JTAG link and checks the design's ID register.

In [ ]:
from nanopynq import Overlay

ol = Overlay("base", board="tangnano4k")
ol

## 2. On, off, toggle

`ol.leds` is the LED GPIO block; the 4K has one LED, `ol.leds[0]`. `read()` returns the value last written (1 = on).

In [ ]:
ol.leds[0].on()
ol.leds[0].read()

In [ ]:
ol.leds[0].off()
ol.leds[0].read()

In [ ]:
ol.leds[0].toggle()
ol.leds[0].read()

## 3. Blink

In [ ]:
import time


def blink(times=5, period=0.4):
    for _ in range(times):
        ol.leds[0].on()
        time.sleep(period / 2)
        ol.leds[0].off()
        time.sleep(period / 2)


blink()

## 4. Follow a button

For 5 seconds the LED copies the first button (`ol.buttons[0]`, pin 14): press it and the LED lights.

In [ ]:
t_end = time.time() + 5
while time.time() < t_end:
    if ol.buttons[0].read():
        ol.leds[0].on()
    else:
        ol.leds[0].off()
    time.sleep(0.02)
ol.leds[0].off()

## 5. The registers underneath

The same LED as a raw register: `ol.ip_dict` lists each block's address; `ol.leds.write(0, 1)` writes the DATA register at offset 0.

In [ ]:
print(ol.ip_dict["leds"]["phys_addr"], ol.ip_dict["leds"]["registers"])
ol.leds.write(0, 1)
ol.leds.read(0)

## 6. Close

Close the link so that the next `Overlay(...)` can open the board. The FPGA keeps running and the LED keeps its state until the board loses power.

In [ ]:
ol.close()